In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Step 1: Load and understand the data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn (1).csv')
print('Dataset shape:', df.shape)
print('\nColumn types:')
print(df.dtypes)
print('\nChurn distribution:')
print(df['Churn'].value_counts())

In [ ]:
# Step 2: Clean the data without discarding usable customers
print('STEP 2: DATA PREPROCESSING')
print('=' * 50)
print('Duplicate rows:', df.duplicated().sum())
print('\nMissing values before cleaning:')
print(df.isna().sum())

# Blank TotalCharges values represent new customers; keep them and impute in the training pipeline.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce')
df = df.drop(columns=['customerID'])
print('\nMissing values after converting TotalCharges:')
print(df.isna().sum())
print('\nCleaned dataset shape:', df.shape)

In [ ]:
# Step 3: Exploratory Data Analysis
print("STEP 3: EXPLORATORY DATA ANALYSIS")
print("="*50)

# 1. Churn distribution
plt.figure(figsize=(8, 5))
churn_counts = df['Churn'].value_counts()
plt.bar(churn_counts.index, churn_counts.values, color=['green', 'red'])
plt.title('Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.show()

# 2. Churn vs Contract type
plt.figure(figsize=(8, 5))
contract_churn = pd.crosstab(df['Contract'], df['Churn'])
contract_churn.plot(kind='bar', color=['green', 'red'])
plt.title('Churn by Contract Type', fontsize=14, fontweight='bold')
plt.xlabel('Contract')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()

# 3. Churn vs Tenure
plt.figure(figsize=(8, 5))
df.boxplot(column='tenure', by='Churn')
plt.title('Tenure vs Churn', fontsize=14, fontweight='bold')
plt.suptitle('')
plt.xlabel('Churn')
plt.ylabel('Tenure (months)')
plt.show()

# 4. Churn vs InternetService
plt.figure(figsize=(8, 5))
internet_churn = pd.crosstab(df['InternetService'], df['Churn'])
internet_churn.plot(kind='bar', color=['green', 'red'])
plt.title('Churn by Internet Service Type', fontsize=14, fontweight='bold')
plt.xlabel('Internet Service')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()

# 5. Summary statistics
print("\nChurn Percentage:")
print(df['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# Step 4: Keep predictors in their original form; fit all transformations on training folds only
print('STEP 4: FEATURE/TARGET SEPARATION')
print('=' * 50)
X = df.drop(columns='Churn')
y = df['Churn'].map({'No': 0, 'Yes': 1})
if y.isna().any():
    raise ValueError('Unexpected target values found in Churn.')
y = y.astype('int8')

categorical_cols = [column for column in X.columns if not pd.api.types.is_numeric_dtype(X[column].dtype)]
numerical_cols = [column for column in X.columns if pd.api.types.is_numeric_dtype(X[column].dtype)]
print('Categorical features:', categorical_cols)
print('Numerical features:', numerical_cols)
print('\nChurn proportions:')
print(y.value_counts(normalize=True).rename(index={0: 'No churn', 1: 'Churn'}))

In [ ]:
# Step 5: Stratified holdout split (kept untouched until final evaluation)
print('STEP 5: TRAIN/TEST SPLIT')
print('=' * 50)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Training churn rate:', round(y_train.mean(), 3))
print('Test churn rate:', round(y_test.mean(), 3))

In [ ]:
# Step 6: Build leakage-safe preprocessing and candidate classifiers
print('STEP 6: MODEL PIPELINES')
print('=' * 50)

numeric_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocessing = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessing, numerical_cols),
    ('categorical', categorical_preprocessing, categorical_cols)
])

def make_pipeline(classifier):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', classifier)
    ])

models = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=1)
    ),
    'Extra Trees': make_pipeline(
        ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=1)
    )
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Preprocessing is fitted separately inside every CV training fold.')

In [ ]:
# Step 7: Compare models using training-only stratified cross-validation
print('STEP 7: CROSS-VALIDATED MODEL COMPARISON')
print('=' * 50)

cv_rows = []
for model_name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv,
        scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc'},
        n_jobs=-1
    )
    cv_rows.append({
        'Model': model_name,
        'CV accuracy mean': scores['test_accuracy'].mean(),
        'CV accuracy std': scores['test_accuracy'].std(),
        'CV churn F1': scores['test_f1'].mean(),
        'CV ROC AUC': scores['test_roc_auc'].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values('CV accuracy mean', ascending=False)
print(cv_results.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
best_model_name = cv_results.iloc[0]['Model']
print(f'\nSelected by mean CV accuracy (not test accuracy): {best_model_name}')

In [ ]:
# Step 7B: Tune the cross-validation winner; optimization uses accuracy, not F1
print('STEP 7B: ACCURACY-FOCUSED TUNING')
print('=' * 50)
print('The untouched test split is reserved for the final one-time evaluation.')

In [ ]:
# Step 8: Tune the selected model and choose a threshold using training-only OOF predictions
print('STEP 8: MODEL TUNING')
print('=' * 50)

if best_model_name == 'Logistic Regression':
    parameter_distributions = {
        'model__C': [0.03, 0.1, 0.3, 1, 3, 10, 30],
        'model__class_weight': [None, 'balanced']
    }
elif best_model_name == 'Random Forest':
    parameter_distributions = {
        'model__n_estimators': [200, 350, 500],
        'model__max_depth': [None, 8, 12, 16, 20],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__class_weight': [None, 'balanced_subsample']
    }
else:
    parameter_distributions = {
        'model__n_estimators': [200, 350, 500],
        'model__max_depth': [None, 8, 12, 16, 20],
        'model__min_samples_split': [2, 5, 10],
        'model__min_samples_leaf': [1, 2, 4],
        'model__max_features': ['sqrt', 0.7, 1.0],
        'model__class_weight': [None, 'balanced']
    }

search = RandomizedSearchCV(
    estimator=models[best_model_name],
    param_distributions=parameter_distributions,
    n_iter=14,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    refit=True,
    verbose=1
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print('\nBest parameters:', search.best_params_)
print(f'Best tuning CV accuracy: {search.best_score_:.4f}')

# Select the probability cutoff from out-of-fold training predictions, never from the test labels.
oof_probabilities = cross_val_predict(
    best_model, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]
thresholds = np.arange(0.20, 0.801, 0.01)
threshold_accuracy = [accuracy_score(y_train, oof_probabilities >= threshold) for threshold in thresholds]
best_threshold = max(
    zip(threshold_accuracy, thresholds),
    key=lambda result: (result[0], -abs(result[1] - 0.5))
)[1]
print(f'OOF-selected decision threshold: {best_threshold:.2f}')

# The test set is evaluated once after model and threshold selection.
test_probabilities = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (test_probabilities >= best_threshold).astype('int8')
test_accuracy = accuracy_score(y_test, y_pred_best)
print(f'\nFinal holdout accuracy: {test_accuracy:.4f} ({test_accuracy:.1%})')
print(f'Accuracy target > 85%: {"MET" if test_accuracy > 0.85 else "NOT MET"}')
print(f'Churn precision: {precision_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'Churn recall:    {recall_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'Churn F1:        {f1_score(y_test, y_pred_best, zero_division=0):.4f}')
print(f'ROC AUC:         {roc_auc_score(y_test, test_probabilities):.4f}')
print('\nConfusion matrix [No churn, Churn]:')
print(confusion_matrix(y_test, y_pred_best))
print('\nClassification report:')
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn'], zero_division=0))

majority_baseline = max(y_test.mean(), 1 - y_test.mean())
print(f'Majority-class baseline accuracy: {majority_baseline:.4f}')

# Export the fitted Assignment 1 pipeline and its training-selected decision threshold.
import joblib

model_bundle = {
    'model': best_model,
    'threshold': float(best_threshold),
    'model_name': str(best_model_name),
    'feature_names': X.columns.tolist(),
}
joblib.dump(model_bundle, 'model.joblib')
print('Saved trained model bundle to model.joblib')

In [ ]:
# Step 9: Score a sample customer with the same fitted preprocessing and threshold
print('STEP 9: SAMPLE PREDICTION')
print('=' * 50)

sample_customer = pd.DataFrame({
    'SeniorCitizen': [0],
    'tenure': [12],
    'MonthlyCharges': [65.50],
    'TotalCharges': [789.00],
    'gender': ['Male'],
    'Partner': ['Yes'],
    'Dependents': ['No'],
    'PhoneService': ['Yes'],
    'MultipleLines': ['Yes'],
    'InternetService': ['Fiber optic'],
    'OnlineSecurity': ['No'],
    'OnlineBackup': ['No'],
    'DeviceProtection': ['No'],
    'TechSupport': ['No'],
    'StreamingTV': ['Yes'],
    'StreamingMovies': ['No'],
    'Contract': ['Month-to-month'],
    'PaperlessBilling': ['Yes'],
    'PaymentMethod': ['Electronic check']
})

sample_probability = best_model.predict_proba(sample_customer)[0, 1]
sample_prediction = int(sample_probability >= best_threshold)
print(f'Estimated churn probability: {sample_probability:.2%}')
print(f'Applied decision threshold: {best_threshold:.2f}')
print('Prediction:', 'CHURN (Yes)' if sample_prediction else 'NO CHURN (No)')